In [ ]:
# Setup on Ibex: locate the repo, make it importable, and report the node.
# No clone and no Drive mount here - the repo lives at $HOME/StarX and the
# data was rsynced in; `git -C $HOME/StarX pull` before starting a session.
import os
import subprocess
import sys


def _find_repo():
    candidates = [globals().get("__vsc_ipynb_file__"), os.getcwd()]
    for start in candidates:
        if not start:
            continue
        path = os.path.abspath(
            os.path.dirname(start) if os.path.isfile(start) else start
        )
        while path != os.path.dirname(path):
            if os.path.exists(os.path.join(path, "starx", "pins.py")):
                return path
            path = os.path.dirname(path)
    home_repo = os.path.join(os.path.expanduser("~"), "StarX")
    if os.path.exists(os.path.join(home_repo, "starx", "pins.py")):
        return home_repo
    raise RuntimeError("could not locate the StarX repo - clone it to ~/StarX")


REPO_DIR = _find_repo()
TRIPOSR_DIR = os.path.join(REPO_DIR, "third_party", "TripoSR")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import torch

print("repo:   ", REPO_DIR)
print("commit: ", subprocess.run(
    ["git", "-C", REPO_DIR, "log", "--oneline", "-1"],
    capture_output=True, text=True).stdout.strip())
print("torch:  ", torch.__version__)
for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}  "
          f"{free / 2**30:.0f}/{total / 2**30:.0f} GiB free")
assert os.path.exists(TRIPOSR_DIR), (
    f"TripoSR clone missing at {TRIPOSR_DIR} - see notebook 08"
)

# 11 - Fine-tuning TripoSR on the sketch dataset (Ibex)

Runs on Ibex, not Colab: no Drive mount, no clone: the repo is already at `$HOME/StarX` and `git pull` brings this notebook up to date. It expects notebook 10 to have built the sketch dataset and that dataset to have reached the node.

This is the paper arm. The model is stock TripoSR with nothing cut or inflated, every parameter trains, and supervision is the rendering loss the report describes and nothing else - no visual-hull term, no auxiliary geometry. The dataset is an ordinary map-style `Dataset` behind a `DataLoader` with workers, so this looks like a normal supervised job rather than the hand-rolled sampler the baseline uses.

The report is five pages and states six training numbers. Those six are reproduced exactly. Everything else it leaves unsaid, and where a value had to come from somewhere the configuration cell marks whether it is the paper's, LRM's, or a choice made for this hardware. The closing section spells out the three places where matching is impossible.

Start it here on one GPU to check the shapes and the memory, then hand the same run to both A100s with the launcher near the end.

In [ ]:
# Configuration - every tunable for this notebook lives here.
# Values marked PAPER are stated in TripoSR's report; the rest are things
# the report is silent on, taken from LRM or fitted to this node.
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from tqdm.auto import tqdm

from starx import cameras, checkpoint, shards, sketchdata
from starx import model as smodel
from starx import train as strain
from starx.config import (
    CAMERA_DISTANCE, FOVY_DEG, StarXConfig, run_dir, shard_dir, sketch_shard_dir,
)

RUN_NAME = "sketch_paper"
RESUME = True
SEED = 1337
EPOCHS = 20
WORKERS = 4
LORA = False    # False: full fine-tuning, as the paper trains. True: adapters.

cfg = StarXConfig(
    drive_root=os.path.join(REPO_DIR, "data", "StarX"),
    local_root=os.path.join(REPO_DIR, "data", "local"),
    # the drawings, and they must match what notebook 10 wrote
    sketch_size=512,
    edge_blur_sigma=1.2,
    edge_gain=3.0,
    edge_bg=1.0,
    # PAPER: the loss and its weights
    lambda_mse=1.0,
    lambda_lpips=2.0,
    lambda_mask=0.05,
    render_crop=128,
    lambda_occ=0.0,        # PAPER: supervision is rendering-only
    # PAPER: the optimizer and schedule. 4e-4 is the paper's value and a
    # from-scratch rate - see the closing section before raising this.
    lr=1e-4,
    warmup_steps=2000,
    # not in the paper: LRM's, or fitted to two A100s
    weight_decay=0.05,
    adam_betas=(0.9, 0.95),
    supervision_views=4,
    batch_designs=8,
    grad_clip=1.0,
    composite_bg=0.5,
    ckpt_every=500,
    val_every=1000,
    keep_k=2,              # full-model checkpoints are large
    gt_size=256,
    eval_chunk=131072,
    seed=SEED,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
AMP_DTYPE, GRAD_SCALE = strain.pick_amp(device)

print(f"run {RUN_NAME} on {torch.cuda.device_count()} visible GPU(s), device {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}   autocast {AMP_DTYPE}")
print(f"loss: MSE + {cfg.lambda_lpips} LPIPS + {cfg.lambda_mask} mask-BCE "
      f"on {cfg.render_crop}px crops, lambda_occ {cfg.lambda_occ}")
print(f"AdamW lr {cfg.lr}, warmup {cfg.warmup_steps}, wd {cfg.weight_decay}, "
      f"betas {cfg.adam_betas}")

In [ ]:
# Open the dataset notebook 10 built and wrap it in a DataLoader. Both
# shard sets extract into one cache; check_params refuses to proceed if the
# stored drawings were made with different edge settings than this config.
train_local = Path(cfg.local_root) / "train"
val_local = Path(cfg.local_root) / "test"
for split, local in (("train", train_local), ("test", val_local)):
    shards.prepare_local(shard_dir(cfg, split), local, progress=tqdm)
    sketch_dir = sketch_shard_dir(cfg, split)
    assert shards.list_done_shards(sketch_dir, sketchdata.SKETCH_PREFIX), (
        f"no sketch shards under {sketch_dir} - run notebook 10 first"
    )
    sketchdata.check_params(sketch_dir, cfg)
    shards.prepare_local(sketch_dir, local, progress=tqdm,
                         prefix=sketchdata.SKETCH_PREFIX)

train_ds = sketchdata.SketchDataset(
    train_local / "cache", supervision_views=cfg.supervision_views, seed=SEED
)
val_ds = sketchdata.SketchDataset(
    val_local / "cache", supervision_views=cfg.supervision_views, seed=SEED
)
loader = torch.utils.data.DataLoader(
    train_ds,
    batch_size=cfg.batch_designs,
    shuffle=True,
    num_workers=WORKERS,
    collate_fn=sketchdata.collate,
    drop_last=True,
    pin_memory=device == "cuda",
    persistent_workers=WORKERS > 0,
)
steps_per_epoch = len(loader)
TOTAL_STEPS = steps_per_epoch * EPOCHS

print(f"train {train_ds.describe()}")
print(f"val   {val_ds.describe()}")
print(f"{steps_per_epoch} steps/epoch x {EPOCHS} epochs = {TOTAL_STEPS} steps")

In [ ]:
# Stock TripoSR, no surgery, every parameter trainable - which is what
# "fine-tune it the way the paper trains it" means. LORA=True swaps in the
# adapter recipe instead: not the paper, but far cheaper and much gentler
# on a checkpoint that is already converged.
model, build_info = smodel.build_stock_lora_model(
    cfg, TRIPOSR_DIR, device=device, full_finetune=not LORA
)
assert model.image_tokenizer.model.embeddings.patch_embeddings.projection.in_channels == 3

model.image_tokenizer.model.gradient_checkpointing_enable()
model.backbone.gradient_checkpointing = True
model.renderer.set_chunk_size(0)

trainable_params = [p for p in model.parameters() if p.requires_grad]
totals = build_info["param_table"]["ALL"]
print(f"trainable: {totals['trainable']:,} of {totals['total']:,} "
      f"({100 * totals['trainable'] / totals['total']:.1f}%)")
if not LORA:
    print(f"checkpoints will hold the full model plus 2x AdamW state: "
          f"~{totals['trainable'] * 12 / 2**30:.1f} GiB each, keep_k={cfg.keep_k}")

In [ ]:
# The optimizer the report specifies: AdamW into a cosine schedule after a
# warmup. Weight decay applies only to matrices and kernels, never to
# biases or normalization scales - that split is LRM's convention, since
# TripoSR states the optimizer but not the decay.
from torchmetrics.image import LearnedPerceptualImagePatchSimilarity

optimizer, scheduler = strain.build_paper_optimizer(model, cfg, TOTAL_STEPS)

lpips_metric = LearnedPerceptualImagePatchSimilarity(net_type="vgg", normalize=True)
lpips_metric = lpips_metric.to(device).requires_grad_(False)
lpips_metric.eval()

decay, no_decay = optimizer.param_groups
print(f"decayed (wd={decay['weight_decay']}): "
      f"{sum(p.numel() for p in decay['params']):,} params")
print(f"undecayed:                {sum(p.numel() for p in no_decay['params']):,} params")
print(f"betas {optimizer.defaults['betas']}   peak lr {cfg.lr}")

# the schedule, drawn - warmup ramp into a cosine that reaches zero
probe = [
    (min((s + 1) / cfg.warmup_steps, 1.0) if s < cfg.warmup_steps
     else 0.5 * (1 + np.cos(np.pi * (s - cfg.warmup_steps)
                            / max(1, TOTAL_STEPS - cfg.warmup_steps)))) * cfg.lr
    for s in range(TOTAL_STEPS)
]
fig, ax = plt.subplots(figsize=(9, 2.4))
ax.plot(probe, color=plt.get_cmap("tab10").colors[4])
ax.axvline(cfg.warmup_steps, ls="--", lw=1, color="0.5")
ax.annotate("end of warmup", (cfg.warmup_steps, cfg.lr * 0.5), fontsize=8)
ax.set_xlabel("step")
ax.set_ylabel("learning rate")
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Resume: restore the newest checkpoint for this run name, or start fresh.
rdir = run_dir(cfg, RUN_NAME)
latest = checkpoint.find_latest(rdir) if RESUME else None
if latest is not None:
    ckpt_path, start_step = latest
    state = checkpoint.load_checkpoint(ckpt_path)
    smodel.load_trainable_state_dict(model, state["model"])
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    checkpoint.restore_rng(state["rng"])
    print(f"resumed {RUN_NAME} at step {start_step}")
else:
    start_step = 0
    print(f"starting {RUN_NAME} fresh")

In [ ]:
# The loop. Resumable: killed and re-run, it picks up from the newest
# checkpoint for this run name.
log_path = rdir / "logs" / "train_log.jsonl"
grid_dir = rdir / "val_grids"
grid_dir.mkdir(parents=True, exist_ok=True)
novel_c2w = cameras.build_spherical_c2w(90.0, 20.0, CAMERA_DISTANCE)
val_indices = [i * val_ds.n_views for i in range(min(4, len(val_ds.designs)))]


def render_val_grid(step_number):
    """Sketch | ground truth | prediction | the same prediction, turned."""
    model.eval()
    model.renderer.set_chunk_size(cfg.eval_chunk)
    rows = [val_ds[i] for i in val_indices]
    fig, axes = plt.subplots(len(rows), 4, figsize=(12.8, 3.1 * len(rows)))
    axes = np.atleast_2d(axes)
    with torch.no_grad():
        for row, item in enumerate(rows):
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=device == "cuda"):
                code = smodel.encode_sketches(model, item["input"][None].to(device))[0]
            code = code.float()
            gt_size = item["views"].shape[1]
            renders = []
            for c2w in (item["c2ws"][0].numpy(), novel_c2w):
                rays_o, rays_d = cameras.rays_full(c2w, FOVY_DEG, gt_size)
                rgb_fg, opacity = strain.render_rays(
                    model, code, rays_o.to(device), rays_d.to(device)
                )
                renders.append(
                    strain.composite_over_gray(rgb_fg, opacity, cfg.composite_bg)
                    .clamp(0, 1).cpu().numpy()
                )
            gt = item["views"][0].numpy().astype(np.float32) / 255.0
            gt = np.where(item["masks"][0].numpy()[..., None], gt, cfg.composite_bg)
            axes[row, 0].imshow(item["input"][0], cmap="gray", vmin=0, vmax=1)
            axes[row, 1].imshow(gt)
            axes[row, 2].imshow(renders[0])
            axes[row, 3].imshow(renders[1])
    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])
    for col, title in enumerate(
        ["sketch (input)", "ground truth", f"prediction @ {step_number}", "turned 90 deg"]
    ):
        axes[0, col].set_title(title, fontsize=10)
    model.renderer.set_chunk_size(0)
    model.train()
    return fig


crop_rng = np.random.default_rng(SEED + 9973 * start_step)
step = start_step
progress = tqdm(total=TOTAL_STEPS, initial=start_step)
while step < TOTAL_STEPS:
    epoch = step // max(1, steps_per_epoch)
    train_ds.set_epoch(epoch)
    for batch in loader:
        if step >= TOTAL_STEPS:
            break
        totals = strain.paper_train_step(
            batch, model, optimizer, scheduler, trainable_params,
            lpips_metric, cfg, device, AMP_DTYPE, crop_rng, GRAD_SCALE,
        )
        step += 1
        progress.update(1)
        progress.set_postfix(loss=f"{totals['loss']:.3f}", epoch=epoch)
        if step % 50 == 0 or step == TOTAL_STEPS:
            checkpoint.append_log(log_path, {"step": step, "epoch": epoch, **totals})
        if step % cfg.ckpt_every == 0 or step == TOTAL_STEPS:
            checkpoint.save_checkpoint(
                rdir, step,
                {
                    "model": smodel.trainable_state_dict(model),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "rng": checkpoint.rng_states(),
                },
                keep_k=cfg.keep_k,
            )
        if step % cfg.val_every == 0 or step == TOTAL_STEPS or step == start_step + 1:
            fig = render_val_grid(step)
            fig.savefig(grid_dir / f"step_{step:07d}.png", dpi=110, bbox_inches="tight")
            plt.show()
progress.close()
print("training finished")

## Scaling to both GPUs

The loop above uses one card. The cell below runs the identical training core on every visible GPU through `scripts/train_sketch.py`, which is the same code path this notebook just used - one process per rank, each drawing its own batch through a `DistributedSampler`, gradients averaged before an identical optimizer step everywhere.

Runs are interchangeable: a run started here can be continued by the script and the other way round, since both write the same run directory format and resume by run name.

In [ ]:
# Both A100s: the same training core as the loop above, launched under
# torchrun as one process per GPU. Each rank draws its own batch through a
# DistributedSampler and gradients are AVERAGED across ranks, so the
# effective batch is nproc x batch_designs at an unchanged learning rate.
# Release the kernel's model first so the ranks get the whole card.
N_GPUS = torch.cuda.device_count()

try:
    del model
    torch.cuda.empty_cache()
    print("released the kernel's model")
except NameError:
    pass

launch = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone", f"--nproc_per_node={N_GPUS}",
    str(Path(REPO_DIR) / "scripts" / "train_sketch.py"),
    "--data-root", cfg.drive_root,
    "--run-name", f"{RUN_NAME}_{N_GPUS}gpu",
    "--lr", str(cfg.lr),
    "--warmup-steps", str(cfg.warmup_steps),
    "--batch-designs", str(cfg.batch_designs),
    "--supervision-views", str(cfg.supervision_views),
    "--epochs", str(EPOCHS),
    "--workers", "4",
]
if LORA:
    launch.append("--lora")

print(f"launching on {N_GPUS} GPU(s) - the same command works in a tmux "
      f"terminal on the node:\n")
print(" ".join(launch), "\n")
subprocess.run(launch, check=False)

## Where this departs from the paper, and why

Three things cannot match, and it is better to name them than to imply otherwise.

**Ground truth is 256px, the paper's is 512px.** Notebook 03 rendered at 256, so a 128px crop covers four times as much of the object as the paper's does - closer to a whole-object view than a detail patch. Re-rendering at 512 is the only real fix, and it is a notebook 03 change, not one that belongs here.

**The learning rate is a from-scratch rate.** 4e-4 was chosen for training from random initialization on roughly a million objects at a batch of a thousand. This run starts from a converged checkpoint and sees a few thousand designs, where that rate will walk the weights a long way from what pretraining found. The configuration cell defaults lower for that reason; set it to the paper's value if faithfulness matters more than the result.

**The batch is a fraction of the paper's.** Theirs was 1024 shapes per iteration across 128 A100s. Two A100s hold a small multiple of eight, so the gradient is noisier per step regardless of how long the run goes.

Everything the report does state is reproduced exactly: the three loss terms and their weights, the mask BCE on rendered opacity, 128px foreground-biased crops, AdamW into a cosine schedule with warmup, no camera conditioning, and supervision that is rendering-only - the visual-hull term from the baseline is off, because the paper says its model relies on rendering losses alone.